## Setup & Configuration


In [2]:
# Install all required packages
import subprocess, sys

packages = [
    "langchain-ollama",
    "langchain-chroma",
    "langchain-community",
    "rank_bm25",
    "pymupdf",
    "pdfplumber",
    "pillow",
    "ragas",
    "datasets",
    "langchain-text-splitters",
    "matplotlib",
    "pandas",
    "requests",
    "pdfplumber",
    "pymupdf",
]

print("Installing packages....")
for pkg in packages:
  r = subprocess.run(
      [sys.executable, "-m", "pip", "install", pkg, "-q"],
      capture_output = True,
      text=True
  )
  status = "Success" if r.returncode == 0 else "error"
  print(f"{status} - {pkg}")
print("All package installed")

Installing packages....
Success - langchain-ollama
Success - langchain-chroma
Success - langchain-community
Success - rank_bm25
Success - pymupdf
Success - pdfplumber
Success - pillow
Success - ragas
Success - datasets
Success - langchain-text-splitters
Success - matplotlib
Success - pandas
Success - requests
Success - pdfplumber
Success - pymupdf
All package installed


In [3]:
import os
import time
import ollama
import fitz

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [4]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 120 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (9,740 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# !rm -f /tmp/ollama-linux-amd64.tar.zst

In [ ]:
# !curl -fL --retry 5 --retry-delay 5 --retry-all-errors https://ollama.com/install.sh -o /tmp/install-ollama.sh

In [ ]:
# !sh /tmp/install-ollama.sh

In [6]:
# Start ollama server
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

In [7]:
!ollama pull qwen3.5:2b
!ollama pull qwen3-embedding:0.6b

In [8]:
LLM_MODEL = "qwen3.5:2b"
EMBED_MODEL = "qwen3-embedding:0.6b"
os.makedirs("./research_pdfs", exist_ok=True)
RESEARCH_PAPERS = [
    {
        "id" : "attention",
        "title" : "Attention Is All You Need",
        "url" : "https://arxiv.org/pdf/1706.03762.pdf",
        "path" : "./research_pdfs/attention.pdf",
        "topic" : "Transformer Architecture"
    },
    {
        "id"    : "bert",
        "title" : "BERT: Pre-training of Deep Bidirectional Transformers",
        "url"   : "https://arxiv.org/pdf/1810.04805",
        "path"  : "./research_pdfs/bert.pdf",
        "topic" : "Language Model Pre-training",
    },
    {
        "id"    : "rag_paper",
        "title" : "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url"   : "https://arxiv.org/pdf/2005.11401",
        "path"  : "./research_pdfs/rag_paper.pdf",
        "topic" : "Retrieval-Augmented Generation",
    }
]

# ----Chroma DB----------------------
CHROMA_PATH = "./chroma_research_db"
COLLECTION_NAME = "research_papers"

# ----Chunking----------------------
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

#----Retrival--------------------------
TOP_K = 5
MAX_RETRIES = 2

print(f"LLM Model : {LLM_MODEL}")
print(f"Embedding Model: {EMBED_MODEL}")
print(f"ChromaDB Path : {CHROMA_PATH}")
print(f"Chunk Size : {CHUNK_SIZE} chars / {CHUNK_OVERLAP} overlap")
print(f"Top-K Retrieval: {TOP_K}")
print(f"\nPapers to index: {len(RESEARCH_PAPERS)}")
for p in RESEARCH_PAPERS:
    print(f"[{p['id']}] {p['title'][:55]}")

LLM Model : qwen3.5:2b
Embedding Model: qwen3-embedding:0.6b
ChromaDB Path : ./chroma_research_db
Chunk Size : 800 chars / 150 overlap
Top-K Retrieval: 5

Papers to index: 3
[attention] Attention Is All You Need
[bert] BERT: Pre-training of Deep Bidirectional Transformers
[rag_paper] Retrieval-Augmented Generation for Knowledge-Intensive 


In [9]:
# Ollama Health Check
import ollama

# List available models
try:
  available = [m.model for m in ollama.list().models]
  print("Available models")
  for m in available:
    print(f" {m}")
except Exception as e:
  print(f"{e}")
  print("Make sure Ollama is running: ollama serve")

# Test LLM
try:
  resp = ollama.chat(
      model = LLM_MODEL,
      messages = [{"role":"user", "content": "Say : LLM ready"}],
      options={"num_predict": 10}
  )
  print(LLM_MODEL)
except Exception as e:
  print(f"{LLM_MODEL}: {e}")
  print(f"Run: ollama pull {LLM_MODEL}")

# Test Embedding
try:
    emb = ollama.embeddings(model=EMBED_MODEL, prompt="test")
    print(f"{EMBED_MODEL} - {len(emb.embedding)}-dim vector")
except Exception as e:
    print(f"{EMBED_MODEL}: {e}")
    print(f"Run: ollama pull {EMBED_MODEL}")

print("\nHealth check complete.")

Available models
 qwen3-embedding:0.6b
 qwen3.5:2b
qwen3.5:2b
qwen3-embedding:0.6b - 1024-dim vector

Health check complete.


In [10]:
# Check Ollama running status
!curl http://localhost:11434

Ollama is running

## PDF Processing PipeLine

In [11]:
# Download PDFs
import requests

def download_pdf(url: str, save_path: str, title: str) -> bool:
  if os.path.exists(save_path):
    size_kb = os.path.getsize(save_path)
    print(f"Already exists: {os.path.basename(save_path)} ({size_kb} KB)")
    return True
  print(f"Downloading {title}")
  headers = {"User-Agent": "Mozilla/5.0 (academic research)"}
  try:
    r = requests.get(url, headers=headers, timeout=90, stream=True)
    r.raise_for_status()
    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
        size_kb = os.path.getsize(save_path)
        print(f"Saved ({size_kb} KB)")
        return True
  except Exception as e:
    print(f"Failed: {e}")
    return False

print("Downloading research papers\n")
success_count = 0
for paper in RESEARCH_PAPERS:
    ok = download_pdf(paper["url"], paper["path"], paper["title"])
    if ok:
        success_count += 1
    time.sleep(1)

print(f"\n{success_count}/{len(RESEARCH_PAPERS)} papers ready")


Saved (2211840 KB)
Saved (775166 KB)
Saved (884736 KB)

3/3 papers ready


In [12]:
# Inspect PDF Structure

for paper in RESEARCH_PAPERS:
    if not os.path.exists(paper["path"]):
        print(f"{paper['title']} - not downloaded")
        continue
    doc = fitz.open(paper["path"])
    total_pages = len(doc)
    total_text_chars = 0
    total_images = 0

    for page in doc:
      total_text_chars += len(page.get_text())
      total_images += len(page.get_images(full=True))

    doc.close()
    print(f"Pages : {total_pages}")
    print(f"Text chars: {total_text_chars:,}")
    print(f"Images : {total_images}")
    print(f"Topic : {paper['topic']}")

Pages : 15
Text chars: 39,498
Images : 3
Topic : Transformer Architecture
Pages : 16
Text chars: 64,117
Images : 34
Topic : Language Model Pre-training
Pages : 19
Text chars: 69,078
Images : 2
Topic : Retrieval-Augmented Generation


In [13]:
# Text Extraction

def extract_text_chunks(paper: dict) -> list:
    if not os.path.exists(paper["path"]):
        return []

    doc    = fitz.open(paper["path"])
    chunks = []
    for page_num, page in enumerate(doc):
        text = page.get_text().strip()
        chunks.append({
            "content"  : text,
            "metadata" : {
                "source" : paper["path"],
                "paper_id" : paper["id"],
                "paper_title": paper["title"],
                "topic" : paper["topic"],
                "page" : page_num + 1,
                "content_type" : "text",
                "char_count" : len(text),
            }
        })
    doc.close()
    return chunks

all_text_chunks = []
print("Extracting text from PDFs\n")

for paper in RESEARCH_PAPERS:
    chunks = extract_text_chunks(paper)
    all_text_chunks.extend(chunks)
    total_chars = sum(c["metadata"]["char_count"] for c in chunks)
    print(f"[{paper['id']:10s}] {len(chunks):2d} pages | {total_chars:,} chars total")

print(f"\nTotal text chunks: {len(all_text_chunks)}")

Extracting text from PDFs

[attention ] 15 pages | 39,483 chars total
[bert      ] 16 pages | 64,101 chars total
[rag_paper ] 19 pages | 69,059 chars total

Total text chunks: 50


In [14]:
# Table Extraction
import pdfplumber

def table_to_markdown(table: list) -> str:
    if not table or len(table) < 2:
        return ""

    rows_md = []
    for row in table:
        cells = [str(c).strip().replace("\n", " ") if c else "" for c in row]
        rows_md.append("| " + " | ".join(cells) + " |")

    # Insert separator after header
    sep = "| " + " | ".join(["---"] * len(table[0])) + " |"
    rows_md.insert(1, sep)
    return "\n".join(rows_md)

def extract_table_chunks(paper: dict) -> list:
    if not os.path.exists(paper["path"]):
        return []

    chunks = []
    with pdfplumber.open(paper["path"]) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for tbl_idx, table in enumerate(tables):
                md = table_to_markdown(table)
                if not md or len(md) < 50:
                    continue

                chunks.append({
                    "content"  : md,
                    "metadata" : {
                        "source"       : paper["path"],
                        "paper_id"     : paper["id"],
                        "paper_title"  : paper["title"],
                        "topic"        : paper["topic"],
                        "page"         : page_num + 1,
                        "table_index"  : tbl_idx,
                        "content_type" : "table",
                        "rows"         : len(table),
                    }
                })
    return chunks

all_table_chunks = []
print("Extracting tables from PDFs\n")
for paper in RESEARCH_PAPERS:
    chunks = extract_table_chunks(paper)
    all_table_chunks.extend(chunks)
    print(f"[{paper['id']:10s}] {len(chunks)} tables found")

print(f"\nTotal table chunks: {len(all_table_chunks)}")
if all_table_chunks:
    sample = all_table_chunks[0]
    print(f"\nSample table (page {sample['metadata']['page']}):")
    print(sample["content"][:500])

Extracting tables from PDFs

[attention ] 8 tables found
[bert      ] 7 tables found
[rag_paper ] 5 tables found

Total table chunks: 20

Sample table (page 9):
| train N d d h d d P ϵ model ff k v drop ls steps |
| --- |
| 6 512 2048 8 64 64 0.1 0.1 100K |
| 1 512 512 4 128 128 16 32 32 32 16 16 |
| 16 32 |
| 2 4 8 256 32 32 1024 128 128 1024 4096 |
| 0.0 0.2 0.0 0.2 |
| positionalembeddinginsteadofsinusoids |
| 6 1024 4096 16 0.3 300K |


In [15]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'camelot-py[cv]', '-q'])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'camelot-py[cv]', '-q'], returncode=0)

In [16]:
import camelot

def extract_tables_camelot(pdf_path: str, page_num: int) -> list[str]:
    try:
        tables = camelot.read_pdf(
            pdf_path,
            pages = str(page_num + 1),
            flavor = 'stream',
            edge_tol = 50,
        )
        results = []
        for t in tables:
            print(f"[Camelot] Page {page_num+1}: accuracy={t.accuracy:.1f}%")
            if t.accuracy < 70:
                print(f"Low confidence — skipping (use Docling or Vision LLM)")
                continue

            md = t.df.to_markdown(index=False)
            results.append(md)
            print(f"Extracted {t.df.shape[0]} rows × {t.df.shape[1]} cols")
            print(f"Preview: {md[:200]}")
        return results
    except ImportError:
        print("  [Camelot] not installed. Run: pip install camelot-py[cv]")
        return []
    except Exception as e:
        print(f"  [Camelot] failed: {e}")
        return []

In [17]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'docling', '-q'])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'docling', '-q'], returncode=0)

In [18]:
def extract_tables_docling(pdf_path: str) -> list[str]:
   try:
      from docling.document_converter import DocumentConverter
      print(f"[Docling] Converting {pdf_path}")

      converter = DocumentConverter()
      result = converter.convert(pdf_path)
      tables = []
      for i, table in enumerate(result.document.tables):
          md = table.export_to_markdown()
          if md and len(md) > 30:
              tables.append(md)
              print(f"[Docling] Table {i+1}: {len(md)} chars")

      print(f"[Docling] Found {len(tables)} tables total")
      return tables

   except ImportError:
        return []
   except Exception as e:
        print(f"[Docling] failed: {e}")
        return []

In [19]:
import io, base64
from PIL import Image

def extract_table_with_vision_llm(
    pdf_path : str,
    page_num : int,
    model    : str = 'qwen2.5vl:3b',
    dpi_scale: float = 2.0
) -> str:
    doc = fitz.open(pdf_path)
    page = doc[page_num]

    mat = fitz.Matrix(dpi_scale, dpi_scale)
    pix = page.get_pixmap(matrix=mat)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    doc.close()

    buf = io.BytesIO()
    img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    prompt = """
    Examine this PDF page carefully.
    Find ALL tables on this page.
    For EACH table:
      1. Extract every row and column exactly as shown
      2. Format it as a markdown table with | separators
      3. Preserve headers, units, and footnotes

    If there are NO tables on this page, respond with exactly: No tables found.

    Output ONLY the markdown tables. No explanation text.
    """
    try:
        resp = ollama.chat(
            model = model,
            messages = [{'role': 'user', 'content': prompt, 'images': [b64]}]
        )
        return resp.message.content.strip()
    except Exception as e:
        return f'Vision LLM error: {e}'

In [20]:
import pdfplumber

def extract_tables_with_fallback(
    pdf_path : str,
    page_num : int,
    use_vision_fallback: bool = False,
    vision_model: str = 'qwen2.5vl:3b'
) -> list[str]:

    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num]
        tables = page.extract_tables()

    if tables:
        results = []
        for t in tables:
            if not t or len(t) < 2:
                continue
            rows_md = []
            for row in t:
                cells = [str(c).strip().replace('\n', ' ') if c else '' for c in row]
                rows_md.append('| ' + ' | '.join(cells) + ' |')
            sep = '| ' + ' | '.join(['---'] * len(t[0])) + ' |'
            rows_md.insert(1, sep)
            md = '\n'.join(rows_md)
            if len(md) > 50:
                results.append(md)
        if results:
            print(f"Page {page_num+1}: pdfplumber found {len(results)} table")
            return results

    results = extract_tables_camelot(pdf_path, page_num)
    if results:
        return results

    if use_vision_fallback:
        print(f"Page {page_num+1}: using Vision LLM fallback")
        vision_result = extract_table_with_vision_llm(
            pdf_path, page_num, model=vision_model
        )
        if 'No tables found' not in vision_result and '|' in vision_result:
            print(f"Page {page_num+1}: Vision LLM extracted table")
            return [vision_result]

    print(f"Page {page_num+1}: no tables found or extraction failed")
    return []


## Handling Images in PDF

In [21]:
def extract_image_metadata(paper: dict, min_px: int = 80) -> list:
    if not os.path.exists(paper["path"]):
        return []

    doc = fitz.open(paper["path"])
    chunks = []
    for page_num, page in enumerate(doc):
        for img_idx, img_info in enumerate(page.get_images(full=True)):
            xref = img_info[0]
            try:
              base_img = doc.extract_image(xref)
              w = base_img["width"]
              h = base_img["height"]

              if w < min_px or h < min_px:
                continue

              content = (
                  f"[Figure {img_idx + 1} on page {page_num + 1} of '{paper['title']}'."
                  f"Image size: {w}×{h} pixels."
                  f"Topic: {paper['topic']}."
                  f"This figure appears in the section covering {paper['topic']}."
                  f"For detailed figure analysis, refer to page {page_num + 1}.]"
                )
              chunks.append({
                    "content" : content,
                    "metadata" : {
                        "source"       : paper["path"],
                        "paper_id"     : paper["id"],
                        "paper_title"  : paper["title"],
                        "topic"        : paper["topic"],
                        "page"         : page_num + 1,
                        "img_index"    : img_idx,
                        "content_type" : "image",
                        "width"        : w,
                        "height"       : h,
                    }
                })
            except Exception:
                continue

    doc.close()
    return chunks

all_image_chunks = []
print("Extracting image metadata from PDFs\n")

for paper in RESEARCH_PAPERS:
    chunks = extract_image_metadata(paper)
    all_image_chunks.extend(chunks)
    print(f"[{paper['id']:10s}] {len(chunks)} images")

print(f"\nTotal image metadata chunks: {len(all_image_chunks)}")
print(f"Text pages : {len(all_text_chunks)}")
print(f"Tables : {len(all_table_chunks)}")
print(f"Images : {len(all_image_chunks)}")
print(f"Total : {len(all_text_chunks) + len(all_table_chunks) + len(all_image_chunks)}")

Extracting image metadata from PDFs

[attention ] 3 images
[bert      ] 34 images
[rag_paper ] 1 images

Total image metadata chunks: 38
Text pages : 50
Tables : 20
Images : 38
Total : 108


In [22]:
VISION_DESCRIBE_PROMPT = """
You are analyzing an image extracted from an AI/ML research paper.

Answer these questions about the image:
1. TYPE: Is it a (diagram / chart / graph / table / photo / equation / other)?
2. CONTENT: What does it show? List all labels, axes, values, and relationships.
3. CONCEPT: What ML/AI concept does it illustrate?
4. KEY NUMBERS: List any important numbers, percentages, or metrics shown.

Be specific and technical. This description will be used for search retrieval,
so include all detail that a researcher might search for.
"""

def describe_image_with_vision_llm(
    pil_image : Image.Image,
    model : str = 'qwen2.5vl:3b',
    page_num : int = 0,
    paper_title : str = ''
) -> str:

    buf = io.BytesIO()
    pil_image.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    try:
       resp = ollama.chat(
          model    = model,
          messages = [{
                'role': 'user',
                'content': VISION_DESCRIBE_PROMPT,
                'images': [b64]
            }]
        )
       description = resp.message.content.strip()
       return (
            f"[Figure on page {page_num} of '{paper_title}'.]\n"
            f"{description}"
        )
    except Exception as e:
        w, h = pil_image.size
        return f"[Figure on page {page_num}: {w}x{h}px. Vision LLM unavailable: {e}]"

def extract_images_with_descriptions(
    paper : dict,
    model : str = 'qwen2.5vl:3b',
    min_px : int = 80,
    max_images: int = 5
) -> list[dict]:
    if not os.path.exists(paper['path']):
        return []

    doc = fitz.open(paper['path'])
    chunks = []
    count = 0

    for page_num, page in enumerate(doc):
        if count >= max_images:
            break

        for img_info in page.get_images(full=True):
            if count >= max_images:
                break

            xref = img_info[0]
            try:
              base_img = doc.extract_image(xref)
              w, h = base_img['width'], base_img['height']

              if w < min_px or h < min_px:
                    continue

              pil_img  = Image.open(io.BytesIO(base_img['image'])).convert('RGB')
              print(f"Describing image {count+1} on page {page_num+1} ({w}x{h})")

              description = describe_image_with_vision_llm(
                  pil_img,
                  model = model,
                  page_num = page_num + 1,
                  paper_title = paper['title']
              )

              chunks.append({
                    'content' : description,
                    'metadata': {
                        'source' : paper['path'],
                        'paper_id' : paper['id'],
                        'paper_title' : paper['title'],
                        'topic': paper['topic'],
                        'page' : page_num + 1,
                        'content_type' : 'image_described',
                        'width': w,
                        'height': h,
                        'extractor' : 'vision_llm',
                    }
                })
              count += 1

            except Exception as e:
                print(f"Skipped: {e}")

    doc.close()
    return chunks

In [23]:
!pip install transformers torch

In [24]:
from transformers import CLIPProcessor, CLIPModel
import torch

def embed_images_with_clip(pil_images: list) -> 'numpy.ndarray':
    model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    inputs = processor(images=pil_images, return_tensors='pt', padding=True)
    with torch.no_grad():
        feats = model.get_image_features(**inputs)

    return feats.numpy()

def embed_texts_with_clip(texts: list) -> 'numpy.ndarray':
    model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        feats = model.get_text_features(**inputs)

    return feats.numpy()

## Chunking

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

sample_text = all_text_chunks[0]["content"] if all_text_chunks else "Sample text not available."
print(f"Input text : {len(sample_text):,} characters\n")

strategies = [
    {
        "name" : "Recursive Character Split (RECOMMENDED)",
        "splitter": RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
    },
]

for s in strategies:
  chunks = s["splitter"].split_text(sample_text)
  sizes = [len(c) for c in chunks]
  print(f"{s['name']}")
  print(f"Chunks : {len(chunks)}")
  print(f"Avg size : {sum(sizes)//len(sizes)} chars")
  print(f"Size range : {min(sizes)}–{max(sizes)} chars")
  print(f"Sample chunk : {chunks[1][:120].strip()}")

Input text : 2,857 characters

Recursive Character Split (RECOMMENDED)
Chunks : 5
Avg size : 644 chars
Size range : 225–784 chars
Sample chunk : convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder


In [26]:
from langchain_core.documents import Document
from collections import Counter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_documents: list[Document] = []
text_split_count = 0
for chunk in all_text_chunks:
    if len(chunk["content"]) > CHUNK_SIZE:
        sub_texts = splitter.split_text(chunk["content"])
        for i, sub in enumerate(sub_texts):
            meta = {**chunk["metadata"], "sub_chunk": i, "total_sub_chunks": len(sub_texts)}
            all_documents.append(Document(page_content=sub, metadata=meta))
        text_split_count += len(sub_texts)
    else:
        all_documents.append(Document(
            page_content=chunk["content"], metadata=chunk["metadata"]
        ))
        text_split_count += 1

for chunk in all_table_chunks + all_image_chunks:
    all_documents.append(Document(
        page_content=chunk["content"], metadata=chunk["metadata"]
    ))

type_counts  = Counter(d.metadata["content_type"] for d in all_documents)
paper_counts = Counter(d.metadata["paper_id"]     for d in all_documents)

print(f"text: {type_counts.get('text', 0)} chunks")
print(f"table: {type_counts.get('table', 0)} chunks")
print(f"image: {type_counts.get('image', 0)} chunks")
print(f"Total: {len(all_documents)} chunks")

for paper_id, count in paper_counts.items():
    print(f"{paper_id}: {count} chunks")

text: 276 chunks
table: 20 chunks
image: 38 chunks
Total: 334 chunks
attention: 74 chunks
bert: 143 chunks
rag_paper: 117 chunks


## Embedding & ChromaDB Vector Store

In [27]:
from langchain_ollama import OllamaEmbeddings
import numpy as np

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
print(f"Embedding model: {EMBED_MODEL}")
test_texts = [
    "The transformer model uses multi-head self-attention.",
    "BERT uses a bidirectional encoder representation.",
    "RAG combines retrieval with generation.",
]

test_embeddings = embeddings.embed_documents(test_texts)
dim = len(test_embeddings[0])
print(f"Embedding dimensions: {dim}")

def cosine_sin(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Transformer - BERT: {cosine_sin(test_embeddings[0], test_embeddings[1]):.3f}  (should be moderate)")
print(f"Transformer - RAG : {cosine_sin(test_embeddings[0], test_embeddings[2]):.3f}  (should be lower)")
print(f"BERT - RAG : {cosine_sin(test_embeddings[1], test_embeddings[2]):.3f}")

Embedding model: qwen3-embedding:0.6b
Embedding dimensions: 1024
Transformer - BERT: 0.667  (should be moderate)
Transformer - RAG : 0.492  (should be lower)
BERT - RAG : 0.623


## Chroma DB

In [28]:
from langchain_chroma import Chroma
import shutil

rebuild = True
if os.path.exists(CHROMA_PATH):
    try:
      existing_db = Chroma(
          collection_name=COLLECTION_NAME,
          embedding_function=embeddings,
          persist_directory=CHROMA_PATH
        )
      existing_count = existing_db._collection.count()
      print(f"Found existing ChromaDB with {existing_count} vectors.")
      if existing_count >= len(all_documents) * 0.9:
          vectorstore = existing_db
          rebuild = False
          print("Using existing database (no re-embedding needed)")
      else:
          print(f"Count mismatch (expected ~{len(all_documents)}). Rebuilding")
          shutil.rmtree(CHROMA_PATH)
    except Exception as e:
        print(f"Cannot load existing DB: {e}. Rebuilding")
        shutil.rmtree(CHROMA_PATH, ignore_errors=True)

if rebuild:
    print(f"Building ChromaDB from {len(all_documents)} chunks")
    batch_size = 50
    batches = [all_documents[i:i+batch_size] for i in range(0, len(all_documents), batch_size)]
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )

    for i, batch in enumerate(batches):
        vectorstore.add_documents(batch)
        print(f"Batch {i+1}/{len(batches)} done ({min((i+1)*batch_size, len(all_documents))}/{len(all_documents)} chunks)")

total = vectorstore._collection.count()
print(f"\nChromaDB ready — {total} vectors stored at {CHROMA_PATH}")

Building ChromaDB from 334 chunks
Batch 1/7 done (50/334 chunks)
Batch 2/7 done (100/334 chunks)
Batch 3/7 done (150/334 chunks)
Batch 4/7 done (200/334 chunks)
Batch 5/7 done (250/334 chunks)
Batch 6/7 done (300/334 chunks)
Batch 7/7 done (334/334 chunks)

ChromaDB ready — 334 vectors stored at ./chroma_research_db


## Verify Retrieval

In [29]:
test_query = "How does multi-head attention work?"

print(f"Query: '{test_query}'")
results = vectorstore.similarity_search_with_score(test_query, k=3)
print("\nTop 3 results (all papers):")
for doc, score in results:
    paper = doc.metadata.get("paper_id", "?")
    page = doc.metadata.get("page", "?")
    ctype = doc.metadata.get("content_type", "?")
    print(f"[{paper} | p{page} | {ctype}] score={score:.4f}")
    print(f"{doc.page_content[:120].strip()}")

filtered = vectorstore.similarity_search(
    test_query, k=2,
    filter={"paper_id": "attention"}
)
for doc in filtered:
    print(f"[p{doc.metadata['page']} | {doc.metadata['content_type']}] {doc.page_content[:100]}")

table_results = vectorstore.similarity_search(
    "model performance BLEU score results", k=3,
    filter={"content_type": "table"}
)
for doc in table_results:
    print(f"[{doc.metadata['paper_id']} | p{doc.metadata['page']}]")
    print(f"{doc.page_content[:200]}")

Query: 'How does multi-head attention work?'

Top 3 results (all papers):
[rag_paper | p1 | text] score=0.6691
combine masked language models [8] with a differentiable retriever, have shown promising results,
arXiv:2005.11401v4  [c
[attention | p5 | text] score=0.6827
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
[attention | p4 | text] score=0.7499
much faster and more space-efficient in practice, since it can be implemented using highly optimized
matrix multiplicati
[p5 | text] output values. These are concatenated and once again projected, resulting in the final values, as
de
[p4 | text] much faster and more space-efficient in practice, since it can be implemented using highly optimized
[bert | p3]
|  |  |  |  |  |  |  | T ’ M |
| --- | --- | --- | --- | --- | --- | --- | --- |
| BERT |  |  |  |  |  |  |  |
| E[CLS] | E 1 | ... | E N | E [SEP] | E’ 1 | ... | E ’ M |
[rag_paper | p8]
| RAG-Tok RAG-Seq FixedDPR 

## RAG Chain

In [30]:
!pip install -U langchain-ollama

In [31]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=8192)
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
    num_ctx=8192,
    num_predict=512,
    reasoning=False
)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a research assistant.
Answer the question using ONLY the provided context.
Always cite which paper and page number your answer comes from.
If the context doesn't contain enough information, say so clearly.

Context (from research papers):
{context}

Question: {question}

Answer (include citations like [paper_title, page X]):
""")

def format_docs(docs):
    parts = []
    for doc in docs:
        paper = doc.metadata.get("paper_title", "Unknown Paper")
        page  = doc.metadata.get("page", "?")
        ctype = doc.metadata.get("content_type", "text").upper()
        parts.append(f"[{ctype} from '{paper}', Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

basic_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print(f"Retriever : top-{TOP_K} chunks by cosine similarity")
print(f"LLM: {LLM_MODEL}")

Retriever : top-5 chunks by cosine similarity
LLM: qwen3.5:2b


In [32]:
# Test Basic RAG

TEST_QUESTIONS = [
    "How many attention heads does the original Transformer model use, and why?",
    "What is the difference between BERT-Base and BERT-Large?",
    "How does RAG handle questions about facts not seen during training?",
]

for q in TEST_QUESTIONS:
    print(f"\nQ: {q}")
    print("-" * 70)

    docs = retriever.invoke(q)
    print("Retrieved chunks:")
    for doc in docs:
        paper = doc.metadata.get("paper_id", "?")
        page  = doc.metadata.get("page", "?")
        ctype = doc.metadata.get("content_type", "?")
        icon  = {"text": "📄", "table": "📊", "image": "🖼️"}.get(ctype, "📄")
        print(f"{icon} [{paper} p{page}] {doc.page_content[:80].strip()}")

    answer = basic_rag_chain.invoke(q)
    print(f"\nAnswer: {answer}")


Q: How many attention heads does the original Transformer model use, and why?
----------------------------------------------------------------------
Retrieved chunks:
📄 [bert p3] the tensor2tensor library.1 Because the use
of Transformers has become common an
🖼️ [attention p4] [Figure 1 on page 4 of 'Attention Is All You Need'.Image size: 445×884 pixels.To
🖼️ [attention p4] [Figure 2 on page 4 of 'Attention Is All You Need'.Image size: 835×1282 pixels.T
🖼️ [attention p3] [Figure 1 on page 3 of 'Attention Is All You Need'.Image size: 1520×2239 pixels.
📄 [attention p5] is similar to that of single-head attention with full dimensionality.
3.2.3
Appl

Answer: Based on the provided context, the specific number of attention heads used by the original Transformer model is not explicitly stated as a fixed constant in the text. However, the paper **Vaswani et al. (2017)** ("Attention Is All You Need") describes the architecture where the number of self-attention heads ($A$) is determined by th

## Advanced RAG

In [33]:
# Multi Query Retriever

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm,
    include_original=True,
)

test_q = "How many attention heads does the transformer use?"
print(f"Original query: '{test_q}'\n")

mq_docs = mq_retriever.invoke(test_q)

print(f"\nRetrieved {len(mq_docs)} unique chunks (vs {TOP_K} from basic RAG):")
for doc in mq_docs:
    paper = doc.metadata.get("paper_id", "?")
    page = doc.metadata.get("page", "?")
    print(f"[{paper} p{page}] {doc.page_content[:90].strip()}")

Original query: 'How many attention heads does the transformer use?'


Retrieved 6 unique chunks (vs 5 from basic RAG):
[bert p3] the tensor2tensor library.1 Because the use
of Transformers has become common and our im-
[attention p4] [Figure 1 on page 4 of 'Attention Is All You Need'.Image size: 445×884 pixels.Topic: Trans
[attention p4] [Figure 2 on page 4 of 'Attention Is All You Need'.Image size: 835×1282 pixels.Topic: Tran
[attention p5] is similar to that of single-head attention with full dimensionality.
3.2.3
Applications o
[attention p3] Figure 1: The Transformer - model architecture.
The Transformer follows this overall archi
[attention p3] [Figure 1 on page 3 of 'Attention Is All You Need'.Image size: 1520×2239 pixels.Topic: Tra


In [34]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": TOP_K}),
)

test_q = "What is the model size (parameters) of BERT-Large?"
print(f"Contextual Compression Test")
print(f"Query: '{test_q}'\n")

print("Without compression (full 800-char chunks):")
raw_docs = vectorstore.similarity_search(test_q, k=2)
for doc in raw_docs:
    print(f"[{doc.metadata['paper_id']} p{doc.metadata['page']}] {len(doc.page_content)} chars")
    print(f"'{doc.page_content[:200].strip()}'")

print("With compression (relevant sentences only):")
compressed_docs = compression_retriever.invoke(test_q)
for doc in compressed_docs:
    print(f"[{doc.metadata['paper_id']} p{doc.metadata['page']}] {len(doc.page_content)} chars")
    print(f"'{doc.page_content.strip()}'")

Contextual Compression Test
Query: 'What is the model size (parameters) of BERT-Large?'

Without compression (full 800-char chunks):
[bert p9] 767 chars
'91.3
12
768
12
3.99
84.4
86.7
92.9
12 1024
16
3.54
85.7
86.9
93.3
24 1024
16
3.23
86.6
87.8
93.7
Table 6:
Ablation over BERT model size. #L = the
number of layers; #H = hidden size; #A = number of at-'
[bert p9] 766 chars
'training data once and then run many experiments
with cheaper models on top of this representation.
In this section, we compare the two approaches
by applying BERT to the CoNLL-2003 Named
Entity Recog'
With compression (relevant sentences only):
[bert p9] 845 chars
'> Question: What is the model size (parameters) of BERT-Large?
> Context:
>>>
91.3
12
768
12
3.99
84.4
86.7
92.9
12 1024
16
3.54
85.7
86.9
93.3
24 1024
16
3.23
86.6
87.8
93.7
Table 6:
Ablation over BERT model size. #L = the
number of layers; #H = hidden size; #A = number of at-
tention heads. "LM (ppl)" is the masked LM perplexity
of held-out training da

In [35]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(all_documents, k=TOP_K)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.6, 0.4],
)

# Compare basic vs hybrid
test_q = "WMT 2014 English-to-German translation BLEU"
print(f"Query: '{test_q}'\n")
print("Basic semantic only:")
basic_docs = vectorstore.similarity_search(test_q, k=3)
for doc in basic_docs:
    print(f"[{doc.metadata['paper_id']} p{doc.metadata['page']}] {doc.page_content}")

print("\nHybrid (BM25 30% + Semantic 70%)")
hybrid_docs = hybrid_retriever.invoke(test_q)
for doc in hybrid_docs[:3]:
    print(f"[{doc.metadata['paper_id']} p{doc.metadata['page']}] {doc.page_content}")

Query: 'WMT 2014 English-to-German translation BLEU'

Basic semantic only:
[rag_paper p1] combine masked language models [8] with a differentiable retriever, have shown promising results,
arXiv:2005.11401v4  [cs.CL]  12 Apr 2021
[attention p1] convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-

/tmp/ipykernel_308/1370045884.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [37]:
from langchain_core.runnables import RunnableLambda

advanced_rag_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

test_q = "What datasets and benchmarks were used to evaluate the BERT model?"
print(f"Question: {test_q}\n")

print("BASIC RAG (semantic only):")
basic_answer = basic_rag_chain.invoke(test_q)
print(basic_answer)

print("\nADVANCED RAG (hybrid BM25 + semantic):")
advanced_answer = advanced_rag_chain.invoke(test_q)
print(advanced_answer)

Question: What datasets and benchmarks were used to evaluate the BERT model?

BASIC RAG (semantic only):
Based on the provided context, the BERT model was evaluated using the following datasets and benchmarks:

*   **CoNLL-2003 Named Entity Recognition (NER)**: Used to compare two approaches for training data. The results are found in Table 6 on Page 9 [paper_title, page 9].
*   **Stanford Sentiment Treebank (SST-2)**: A binary single-sentence classification task used as part of the GLUE benchmark evaluation.
*   **Stanford Question Answering Dataset (SQuAD v1.1)**: A collection of 100k crowd-sourced question/answer pairs used for evaluation.
*   **GLUE Leaderboard**: The official benchmark where BERTLARGE obtained a score of 80.5 and BERTBASE obtained 76.8 (Page 6).
*   **Other GLUE Tasks**: Specifically QQP, MRPC, STS-B, and SST-2, with F1 scores reported for the former two and Spearman correlations for the latter [paper_title, page 6].

**Answer:**
The BERT model was evaluated on th

## Agentic RAG with LangGraph (CRAG Pattern)

In [38]:
from typing import TypedDict, List, Optional
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class RAGState(TypedDict):
    question : str
    generation : str
    documents : List[Document]
    steps : List[str]
    retries : int
    gen_retries: int
    route : str

grader_llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=1024, num_predict=5)
GRADE_PROMPT = ChatPromptTemplate.from_template("""
Answer ONLY 'yes' or 'no'.

{context}

Answer (yes/no):
""")

grade_chain = GRADE_PROMPT | grader_llm | StrOutputParser()

ROUTER_PROMPT = ChatPromptTemplate.from_template("""
Route this question. Answer ONLY 'vectorstore' or 'direct'.
- 'vectorstore': questions about AI papers (transformers, BERT, RAG, attention)
- 'direct': greetings, simple math, unrelated topics

Question: {question}

Answer (vectorstore/direct):
""")

router_llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=512, num_predict=5)
router_chain = ROUTER_PROMPT | router_llm | StrOutputParser()

In [39]:
from langchain_core.messages import HumanMessage, ToolMessage

GRADE_DOCS = False

def route_query(state: RAGState) -> RAGState:
    """Decide: retrieve from vectorstore or answer directly."""
    question = state["question"]
    route = router_chain.invoke({"question": question}).strip().lower()
    route = "direct" if "direct" in route else "vectorstore"
    return {**state,
            "route": route,
            "steps": state.get("steps", []) + [f"route→{route}"]}


def retrieve(state: RAGState) -> RAGState:
    """Retrieve relevant documents using hybrid search."""
    docs = hybrid_retriever.invoke(state["question"])
    return {**state,
            "documents": docs,
            "steps": state["steps"] + [f"retrieve({len(docs)} docs)"]}


def grade_documents(state: RAGState) -> RAGState:
    """
    Filter docs for relevance — each doc costs 1 LLM call.
    SKIPPED when GRADE_DOCS=False (default) for speed.
    """
    if not GRADE_DOCS:
        return {**state,
                "steps": state["steps"] + ["grade(skipped-using-all-docs)"]}

    question = state["question"]
    filtered = []
    for doc in state["documents"]:
        ctx = f"Question: {question}\nDocument: {doc.page_content[:300]}\nRelevant?"
        score = grade_chain.invoke({"context": ctx}).strip().lower()
        if "yes" in score:
            filtered.append(doc)

    kept = filtered if filtered else state["documents"]   # fallback: keep all if none pass
    return {**state,
            "documents": kept,
            "steps": state["steps"] + [f"grade({len(kept)}/{len(state['documents'])} kept)"]}


def generate(state: RAGState) -> RAGState:
    """Generate answer from retrieved documents."""
    from langchain_core.output_parsers import StrOutputParser
    docs = state.get("documents", [])
    context = format_docs(docs) if docs else "No documents retrieved."
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": context, "question": state["question"]
    })

    return {**state,
            "generation" : answer,
            "gen_retries": state.get("gen_retries", 0) + 1,
            "steps" : state["steps"] + ["generate"]}


def transform_query(state: RAGState) -> RAGState:
    """Rewrite query to improve retrieval when graded docs were irrelevant."""

    rewrite_prompt = ChatPromptTemplate.from_template("""
    Rewrite this question using different technical terms for better search retrieval.
    Return ONE line only.
    Original: {question}
    Rewritten:
    """)
    new_q = (rewrite_prompt | llm | StrOutputParser()).invoke(
        {"question": state["question"]}
    ).strip().split("\n")[0]

    return {**state,
            "question": new_q,
            "retries" : state.get("retries", 0) + 1,
            "steps" : state["steps"] + [f"transform→'{new_q[:45]}...'"]}


def check_hallucination(state: RAGState) -> RAGState:
    """Verify the answer is grounded in retrieved docs."""
    docs = state.get("documents", [])[:2]
    generation = state.get("generation", "")
    ctx = (
        f"Documents:\n{format_docs(docs)}\n\n"
        f"Answer:\n{generation[:400]}\n\n"
        f"Is the answer grounded in the documents? (yes/no)"
    )
    score = grade_chain.invoke({"context": ctx}).strip().lower()
    result = "grounded" if "yes" in score else "hallucinated"
    return {**state,
            "steps": state["steps"] + [f"hallucination→{result}"]}


def decide_route(state: RAGState) -> str:
    return state.get("route", "vectorstore")


def decide_after_grading(state: RAGState) -> str:
    """After grading: generate if docs exist, else transform query (max retries)."""
    if state.get("documents") or state.get("retries", 0) >= MAX_RETRIES:
        return "generate"
    return "transform_query"


def decide_hallucination(state: RAGState) -> str:
    """
    FIX: Exit if grounded OR if gen_retries >= MAX_RETRIES.
    Before fix: gen_retries was never incremented → infinite loop.
    After fix:  gen_retries++ in generate() → loop exits after MAX_RETRIES.
    """
    steps = state.get("steps", [])
    gen_retries = state.get("gen_retries", 0)
    last_step= steps[-1] if steps else ""

    if "grounded" in last_step or gen_retries >= MAX_RETRIES:
        return "END"
    return "generate"

print(f"GRADE_DOCS = {GRADE_DOCS} ")
print(f"MAX_RETRIES = {MAX_RETRIES}")

GRADE_DOCS = False 
MAX_RETRIES = 2


In [40]:
from langgraph.graph import StateGraph, END

def build_rag_graph():
    workflow = StateGraph(RAGState)

    workflow.add_node("route_query", route_query)
    workflow.add_node("retrieve", retrieve)
    workflow.add_node("grade_documents", grade_documents)
    workflow.add_node("generate", generate)
    workflow.add_node("transform_query", transform_query)
    workflow.add_node("check_hallucination", check_hallucination)
    workflow.set_entry_point("route_query")
    workflow.add_conditional_edges(
        "route_query", decide_route,
        {"vectorstore": "retrieve", "direct": "generate"}
    )
    workflow.add_edge("retrieve", "grade_documents")
    workflow.add_conditional_edges(
        "grade_documents", decide_after_grading,
        {"generate": "generate", "transform_query": "transform_query"}
    )
    workflow.add_edge("transform_query", "retrieve")
    workflow.add_edge("generate", "check_hallucination")
    workflow.add_conditional_edges(
        "check_hallucination", decide_hallucination,
        {"END": END, "generate": "generate"}
    )

    return workflow.compile()

rag_app = build_rag_graph()
print("Agentic RAG graph compiled")

Agentic RAG graph compiled


In [41]:
import time

def run_rag_agent(question: str, verbose: bool = True) -> dict:
    """Run the self-correcting RAG agent and return final state."""
    initial_state = {
        "question" : question,
        "generation" : "",
        "documents" : [],
        "steps" : [],
        "retries" : 0,
        "gen_retries": 0,
        "route" : "",
    }

    t = time.time()
    final_state = rag_app.invoke(initial_state)
    elapsed = time.time() - t

    if verbose:
        print(f"Q: {question}")
        print(f"\nAgent Steps ({elapsed:.1f}s): {' → '.join(final_state['steps'])}")
        print(f"\nSources used:")
        for doc in final_state.get("documents", []):
            paper = doc.metadata.get("paper_id", "?")
            page = doc.metadata.get("page", "?")
            ctype = doc.metadata.get("content_type", "?")
            print(f"[{paper} p{page} | {ctype}]")
        print(f"\nAnswer:\n{final_state['generation']}")

    return final_state
result = run_rag_agent("What is the key innovation of the Transformer model over RNNs?")
print()

Q: What is the key innovation of the Transformer model over RNNs?

Agent Steps (19.4s): route→vectorstore → retrieve(9 docs) → grade(skipped-using-all-docs) → generate → hallucination→hallucinated → generate → hallucination→hallucinated

Sources used:
[attention p2 | text]
[attention p6 | text]
[attention p9 | text]
[bert p3 | text]
[rag_paper p9 | text]
[attention p10 | text]
[attention p1 | text]
[attention p9 | text]
[attention p2 | text]

Answer:
Based on the provided context, the key innovation of the Transformer model over Recurrent Neural Networks (RNNs) is that it **eschews recurrence and relies entirely on an attention mechanism to draw global dependencies between input and output sequences**, rather than using sequence-aligned recurrence [1].

This approach allows the Transformer to model dependencies without regard to their distance in the input or output sequences, significantly improving performance on tasks like translation and language modeling compared to models that us

## RAG Evaluation with RAGAS

In [42]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

from langchain_ollama import ChatOllama

ragas_llm = LangchainLLMWrapper(
    ChatOllama(
        model=LLM_MODEL,
        temperature=0,
        format="json",
        num_ctx=8192,
        num_predict=2048,
        reasoning=False,   # Disable thinking mode
    )
)
ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL))

# Initialize metrics with our local models
faithfulness  = Faithfulness(llm=ragas_llm)
answer_relevancy   = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb)
context_precision  = LLMContextPrecisionWithReference(llm=ragas_llm)
context_recall = LLMContextRecall(llm=ragas_llm)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

print(f"LLM Judge : {LLM_MODEL} (via Ollama)")
print(f"Embeddings : {EMBED_MODEL} (via Ollama)")
print(f"Metrics : {len(metrics)}")
for m in metrics:
    print(f"{m.__class__.__name__}")

LLM Judge : qwen3.5:2b (via Ollama)
Embeddings : qwen3-embedding:0.6b (via Ollama)
Metrics : 4
Faithfulness
AnswerRelevancy
LLMContextPrecisionWithReference
LLMContextRecall


/tmp/ipykernel_308/3758887437.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_308/3758887437.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/tmp/ipykernel_308/3758887437.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecisionWithReference
  from ragas.metrics import (
/tmp/ipykernel_308/3758887437.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is depreca

In [44]:
EVAL_SAMPLES = [
    {
        "question"    : "How many attention heads does the base Transformer model use?",
        "ground_truth": "The base Transformer model uses 8 attention heads (h=8) with a model dimension of 512.",
    },
    {
        "question"    : "What are the two pre-training tasks used in BERT?",
        "ground_truth": "BERT uses two pre-training tasks: Masked Language Model (MLM) where 15% of tokens are masked and predicted, and Next Sentence Prediction (NSP) where the model predicts if sentence B follows sentence A.",
    },
    {
        "question"    : "What is the key advantage of RAG over purely parametric models?",
        "ground_truth": "RAG can access and update non-parametric memory (retrieved documents) without retraining, allowing it to handle knowledge-intensive tasks using up-to-date information from a document store.",
    },
    {
        "question"    : "What optimizer was used to train the original Transformer?",
        "ground_truth": "The original Transformer used the Adam optimizer with custom learning rate scheduling that increases linearly for the first warmup steps then decreases proportionally to the inverse square root of the step number.",
    },
]

eval_data = []
for sample in EVAL_SAMPLES:
    q = sample["question"]
    retrieved_docs = hybrid_retriever.invoke(q)
    contexts = [doc.page_content for doc in retrieved_docs]
    answer = advanced_rag_chain.invoke(q)
    eval_data.append({
        "user_input" : q,
        "retrieved_contexts" : contexts,
        "response" : answer,
        "reference" : sample["ground_truth"],
    })

In [45]:
from ragas import evaluate
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
import pandas as pd
import math

ragas_samples = [
    SingleTurnSample(
        user_input = d["user_input"],
        retrieved_contexts = d["retrieved_contexts"],
        response = d["response"],
        reference = d["reference"],
    )
    for d in eval_data
]

ragas_dataset = EvaluationDataset(samples=ragas_samples)
results = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    run_config=RunConfig(timeout=600, max_workers=1),
)

df = results.to_pandas()
score_cols = [
    col for col in df.columns
    if col not in ["user_input", "retrieved_contexts", "response", "reference"]
]

for i, row in df.iterrows():

    print(f"\nQUESTION {i+1}:")
    print(row["user_input"])

    print("\nRETRIEVED CONTEXTS:")
    for context in row["retrieved_contexts"]:
        print(context)

    print("\nGENERATED RESPONSE:")
    print(row["response"])

    print("\nREFERENCE ANSWER:")
    print(row["reference"])

    print("\nEVALUATION SCORES:")
    for col in score_cols:
        print(f"{col}: {row[col]}")

    print("-" * 50)

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]


QUESTION 1:
How many attention heads does the base Transformer model use?

RETRIEVED CONTEXTS:
the tensor2tensor library.1 Because the use
of Transformers has become common and our im-
plementation is almost identical to the original,
we will omit an exhaustive background descrip-
tion of the model architecture and refer readers to
Vaswani et al. (2017) as well as excellent guides
such as “The Annotated Transformer.”2
In this work, we denote the number of layers
(i.e., Transformer blocks) as L, the hidden size as
H, and the number of self-attention heads as A.3
We primarily report results on two model sizes:
BERTBASE (L=12, H=768, A=12, Total Param-
eters=110M) and BERTLARGE (L=24, H=1024,
A=16, Total Parameters=340M).
BERTBASE was chosen to have the same model
size as OpenAI GPT for comparison purposes.
Critically, however, the BERT Transformer uses
256
32
32
5.75
24.5
28
1024
128
128
4.66
26.0
168
1024
5.12
25.4
53
4096
4.75
26.2
90
(D)
0.0
5.77
24.6
0.2
4.95
25.5
0.0
4.67
25.3
0.2


In [46]:
from IPython.display import display

metrics_df = df[[
    "user_input",
    "faithfulness",
    "answer_relevancy",
    "llm_context_precision_with_reference",
    "context_recall"
]].copy()

metrics_df.columns = [
    "Question",
    "Faithfulness",
    "Answer Relevancy",
    "Context Precision",
    "Context Recall"
]

metrics_df["Question"] = metrics_df["Question"].str.slice(0, 65)
metric_columns = [
    "Faithfulness",
    "Answer Relevancy",
    "Context Precision",
    "Context Recall"
]

metrics_df[metric_columns] = metrics_df[metric_columns].round(3)

display(
    metrics_df.style
    .format({col: "{:.3f}" for col in metric_columns})
    .background_gradient(subset=metric_columns, cmap="RdYlGn")
    .set_properties(**{"text-align": "center"})
    .set_caption("NexaRAG - RAGAS Evaluation Results")
)

,Question,Faithfulness,Answer Relevancy,Context Precision,Context Recall
0,How many attention heads does the base Transformer model use?,0.600,0.973,0.500,0.000
1,What are the two pre-training tasks used in BERT?,0.500,0.987,0.433,1.000
2,What is the key advantage of RAG over purely parametric models?,0.833,0.951,0.810,1.000
3,What optimizer was used to train the original Transformer?,0.400,0.937,0.500,1.000


In [47]:
retrievers = {
    "Semantic": retriever,
    "BM25": bm25_retriever,
    "Hybrid": hybrid_retriever
}

test_questions = [
    "What datasets and benchmarks were used to evaluate the BERT model?",
    "How does the Transformer use self-attention?",
    "What are the advantages of Retrieval Augmented Generation?"
]

K = TOP_K

def get_page_reference(doc):
    title = doc.metadata.get("paper_title", "Unknown")
    page = doc.metadata.get("page", "Unknown")
    return (title, page)


results = []

for question in test_questions:

    print(f"\nQUESTION: {question}")

    for name, retriever_obj in retrievers.items():

        start = time.time()
        docs = retriever_obj.invoke(question)
        latency = time.time() - start

        print(f"\n{name} RETRIEVER")

        for rank, doc in enumerate(docs[:K], start=1):
            print(f"\nRank: {rank}")
            print("Paper:", doc.metadata.get("paper_title"))
            print("Page:", doc.metadata.get("page"))
            print("Chunk:", doc.page_content[:250])

        results.append({
            "Question": question,
            "Retriever": name,
            "Retrieved Chunks": len(docs[:K]),
            "Latency (seconds)": round(latency, 3)
        })

comparison_df = pd.DataFrame(results)

display(comparison_df)


QUESTION: What datasets and benchmarks were used to evaluate the BERT model?

Semantic RETRIEVER

Rank: 1
Paper: BERT: Pre-training of Deep Bidirectional Transformers
Page: 9
Chunk: training data once and then run many experiments
with cheaper models on top of this representation.
In this section, we compare the two approaches
by applying BERT to the CoNLL-2003 Named
Entity Recognition (NER) task (Tjong Kim Sang
and De Meulder, 

Rank: 2
Paper: BERT: Pre-training of Deep Bidirectional Transformers
Page: 15
Chunk: BERT
E[CLS]
E1
 E[SEP]
...
EN
E1’
...
EM’
C
T1
T[SEP]
...
TN
T1’
...
TM’
[CLS]
Tok 
1
 [SEP]
...
Tok 
N
Tok 
1
...
Tok
M
Question
Paragraph
BERT
E[CLS]
E1
 E2
 EN
C
T1
 T2
 TN
Single Sentence 
...
...
BERT
Tok 1
 Tok 2
 Tok N
...
[CLS]
E[CLS]
E1
 E2


Rank: 3
Paper: BERT: Pre-training of Deep Bidirectional Transformers
Page: 3
Chunk: |  |  |  |  |  |  |  | T ’ M |
| --- | --- | --- | --- | --- | --- | --- | --- |
| BERT |  |  |  |  |  |  |  |
| E[CLS] | E 1 | ... | E N | E [S

,Question,Retriever,Retrieved Chunks,Latency (seconds)
0,What datasets and benchmarks were used to eval...,Semantic,5,0.028
1,What datasets and benchmarks were used to eval...,BM25,5,0.002
2,What datasets and benchmarks were used to eval...,Hybrid,5,0.026
3,How does the Transformer use self-attention?,Semantic,5,0.022
4,How does the Transformer use self-attention?,BM25,5,0.001
5,How does the Transformer use self-attention?,Hybrid,5,0.024
6,What are the advantages of Retrieval Augmented...,Semantic,5,0.021
7,What are the advantages of Retrieval Augmented...,BM25,5,0.001
8,What are the advantages of Retrieval Augmented...,Hybrid,5,0.024


In [48]:
K = TOP_K
ground_truth = {
    "What datasets and benchmarks were used to evaluate the BERT model?": [
        ("BERT: Pre-training of Deep Bidirectional Transformers", 6),
        ("BERT: Pre-training of Deep Bidirectional Transformers", 7),
        ("BERT: Pre-training of Deep Bidirectional Transformers", 9)
    ],

    "How does the Transformer use self-attention?": [
        ("Attention Is All You Need", 3),
        ("Attention Is All You Need", 5)
    ],

    "What are the advantages of Retrieval Augmented Generation?": [
        ("Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", 2),
        ("Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", 3),
        ("Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", 7)
    ]
}

def calculate_metrics(retrieved_docs, relevant_docs, k=TOP_K):

    retrieved = [
        (
            doc.metadata.get("paper_title"),
            doc.metadata.get("page")
        )
        for doc in retrieved_docs[:k]
    ]

    relevant_retrieved = sum(
        1 for doc in retrieved
        if doc in relevant_docs
    )

    precision = relevant_retrieved / k
    recall = relevant_retrieved / len(relevant_docs)
    reciprocal_rank = 0
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant_docs:
            reciprocal_rank = 1 / rank
            break

    return precision, recall, reciprocal_rank

results = []

for question in test_questions:
    relevant_docs = ground_truth[question]
    for name, retriever_obj in retrievers.items():

        start = time.time()
        docs = retriever_obj.invoke(question)
        latency = time.time() - start

        precision, recall, mrr = calculate_metrics(
            docs,
            relevant_docs,
            K
        )

        results.append({
            "Question": question,
            "Retriever": name,
            "Precision@5": round(precision, 3),
            "Recall@5": round(recall, 3),
            "MRR": round(mrr, 3),
            "Latency (seconds)": round(latency, 3)
        })

metrics_df = pd.DataFrame(results)
display(metrics_df)

,Question,Retriever,Precision@5,Recall@5,MRR,Latency (seconds)
0,What datasets and benchmarks were used to eval...,Semantic,0.6,1.000,1.00,0.025
1,What datasets and benchmarks were used to eval...,BM25,0.0,0.000,0.00,0.002
2,What datasets and benchmarks were used to eval...,Hybrid,0.0,0.000,0.00,0.025
3,How does the Transformer use self-attention?,Semantic,0.4,1.000,1.00,0.021
4,How does the Transformer use self-attention?,BM25,0.0,0.000,0.00,0.001
5,How does the Transformer use self-attention?,Hybrid,0.0,0.000,0.00,0.024
6,What are the advantages of Retrieval Augmented...,Semantic,0.4,0.667,0.25,0.023
7,What are the advantages of Retrieval Augmented...,BM25,0.2,0.333,1.00,0.001
8,What are the advantages of Retrieval Augmented...,Hybrid,0.2,0.333,1.00,0.025


In [ ]:
def calculate_retrieval_metrics(retriever, eval_samples, k=5):

    precision_scores = []
    recall_scores = []
    reciprocal_ranks = []

    for sample in eval_samples:

        question = sample["question"]
        ground_truth = sample["ground_truth"].lower()
        retrieved_docs = retriever.invoke(question)[:k]
        keywords = [
            word for word in ground_truth.split()
            if len(word) > 4
        ]

        relevant = []
        for doc in retrieved_docs:
            content = doc.page_content.lower()
            is_relevant = any(
                keyword in content for keyword in keywords
            )

            relevant.append(is_relevant)

        precision_scores.append(sum(relevant) / k)
        recall_scores.append(float(any(relevant)))
        reciprocal_rank = 0
        for rank, is_relevant in enumerate(relevant, start=1):
            if is_relevant:
                reciprocal_rank = 1 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

    return {
        "Precision@5": sum(precision_scores) / len(precision_scores),
        "Recall@5": sum(recall_scores) / len(recall_scores),
        "MRR": sum(reciprocal_ranks) / len(reciprocal_ranks)
    }

In [ ]:
ragas_metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

ragas_score_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

all_results = []
individual_results = {}

for name, retriever_obj in retrievers.items():

    print(f"\nEvaluating Retriever: {name}")
    eval_data = []
    for sample in EVAL_SAMPLES:

        q = sample["question"]
        retrieved_docs = retriever_obj.invoke(q)
        contexts = [doc.page_content for doc in retrieved_docs]
        answer = (
            RAG_PROMPT
            | llm
            | StrOutputParser()
        ).invoke({
            "context": format_docs(retrieved_docs),
            "question": q
        })

        eval_data.append({
            "user_input": q,
            "retrieved_contexts": contexts,
            "response": answer,
            "reference": sample["ground_truth"]
        })

    ragas_samples = [
        SingleTurnSample(
            user_input=d["user_input"],
            retrieved_contexts=d["retrieved_contexts"],
            response=d["response"],
            reference=d["reference"]
        )
        for d in eval_data
    ]

    ragas_dataset = EvaluationDataset(samples=ragas_samples)
    results = evaluate(
        dataset=ragas_dataset,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_emb,
        run_config=RunConfig(
            timeout=600,
            max_workers=1
        )
    )

    df = results.to_pandas()
    individual_results[name] = df
    avg_ragas = df[ragas_score_cols].mean()
    print(f"\n{name} - RAGAS Scores")
    display(df)

    retrieval_scores = calculate_retrieval_metrics(
        retriever_obj,
        EVAL_SAMPLES
    )

    all_results.append({
        "Retriever": name,
        "Precision@5": retrieval_scores["Precision@5"],
        "Recall@5": retrieval_scores["Recall@5"],
        "MRR": retrieval_scores["MRR"],
        "Faithfulness": avg_ragas["faithfulness"],
        "Answer Relevancy": avg_ragas["answer_relevancy"],
        "Context Precision": avg_ragas["context_precision"],
        "Context Recall": avg_ragas["context_recall"]
    })

comparison_df = pd.DataFrame(all_results).set_index("Retriever")

display(comparison_df.round(4))
retrieval_cols = ["Precision@5", "Recall@5", "MRR"]

ragas_cols = [
    "Faithfulness",
    "Answer Relevancy",
    "Context Precision",
    "Context Recall"
]

comparison_df["Retrieval Score"] = comparison_df[retrieval_cols].mean(axis=1)
comparison_df["RAGAS Score"] = comparison_df[ragas_cols].mean(axis=1)

comparison_df["Overall Score"] = (
    comparison_df["Retrieval Score"] * 0.30 +
    comparison_df["RAGAS Score"] * 0.70
)

comparison_df = comparison_df.sort_values(
    "Overall Score",
    ascending=False
)

display(comparison_df.round(4))
print("Best Retriever:", comparison_df["Overall Score"].idxmax())
print("Best Overall Score:", round(comparison_df["Overall Score"].max(), 4))



Evaluating Retriever: Semantic


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]


Semantic - RAGAS Scores


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,How many attention heads does the base Transfo...,[the tensor2tensor library.1 Because the use\n...,The base Transformer model uses 12 attention h...,The base Transformer model uses 8 attention he...,0.000000,0.896327,0.000000,0.0
1,What are the two pre-training tasks used in BERT?,[BERT\nBERT\nE[CLS]\nE1\n E[SEP]\n...\nEN\nE1’...,"Based on the provided context, the two pre-tra...",BERT uses two pre-training tasks: Masked Langu...,0.833333,0.987113,0.833333,1.0
2,What is the key advantage of RAG over purely p...,[memory. We showed that our RAG models obtain ...,"Based on the provided context, the key advanta...",RAG can access and update non-parametric memor...,1.000000,0.951337,0.755556,1.0
3,What optimizer was used to train the original ...,[our model establishes a new single-model stat...,The provided context does not contain informat...,The original Transformer used the Adam optimiz...,0.600000,0.000000,0.000000,0.0



Evaluating Retriever: BM25


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

... (6 outputs left out)

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,How many attention heads does the base Transfo...,[the tensor2tensor library.1 Because the use\n...,The base Transformer model described in the co...,The base Transformer model uses 8 attention he...,0.750000,0.976319,0.000000,0.0
1,What are the two pre-training tasks used in BERT?,[BERT\nBERT\nE[CLS]\nE1\n E[SEP]\n...\nEN\nE1’...,"Based on the provided context, the two pre-tra...",BERT uses two pre-training tasks: Masked Langu...,0.400000,1.000000,0.610417,1.0
2,What is the key advantage of RAG over purely p...,[memory. We showed that our RAG models obtain ...,"Based on the provided context, the key advanta...",RAG can access and update non-parametric memor...,1.000000,0.951337,0.755556,1.0
3,What optimizer was used to train the original ...,[previous state-of-the-art model. The Transfor...,The original Transformer model was trained usi...,The original Transformer used the Adam optimiz...,0.857143,0.895589,0.166667,1.0


,Precision@5,Recall@5,MRR,Faithfulness,Answer Relevancy,Context Precision,Context Recall
Retriever,,,,,,,
Semantic,0.9,1.0,1.0,0.6083,0.7087,0.3972,0.50
BM25,1.0,1.0,1.0,0.5926,0.9469,0.5917,0.75
Hybrid,0.9,1.0,1.0,0.7518,0.9558,0.3832,0.75


,Precision@5,Recall@5,MRR,Faithfulness,Answer Relevancy,Context Precision,Context Recall,Retrieval Score,RAGAS Score,Overall Score
Retriever,,,,,,,,,,
BM25,1.0,1.0,1.0,0.5926,0.9469,0.5917,0.75,1.0000,0.7203,0.8042
Hybrid,0.9,1.0,1.0,0.7518,0.9558,0.3832,0.75,0.9667,0.7102,0.7871
Semantic,0.9,1.0,1.0,0.6083,0.7087,0.3972,0.50,0.9667,0.5536,0.6775


Best Retriever: BM25
Best Overall Score: 0.8042


In [ ]:
weight_combinations = [
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3)
]

weight_results = []

for bm25_weight, semantic_weight in weight_combinations:

    print(f"\nTesting weights: BM25={bm25_weight}, Semantic={semantic_weight}")

    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, semantic_retriever],
        weights=[bm25_weight, semantic_weight]
    )

    eval_data = []

    for d in EVAL_SAMPLES:

        retrieved_docs = hybrid_retriever.invoke(d["question"])

        response = (RAG_PROMPT | llm | StrOutputParser()).invoke({
            "context": format_docs(retrieved_docs),
            "question": d["question"]
        })


        eval_data.append({
            "user_input": d["question"],
            "retrieved_contexts": [doc.page_content for doc in retrieved_docs],
            "response": response,
            "reference": d["ground_truth"]
        })

    # Reuse your existing RAGAS evaluation setup
    ragas_samples = [
        SingleTurnSample(
            user_input=d["user_input"],
            retrieved_contexts=d["retrieved_contexts"],
            response=d["response"],
            reference=d["reference"]
        )
        for d in eval_data
    ]

    ragas_dataset = EvaluationDataset(samples=ragas_samples)

    results = evaluate(
    dataset=ragas_dataset,
    metrics=ragas_metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=RunConfig(timeout=600, max_workers=1)
)

    # results = evaluate(
    #     dataset=ragas_dataset,
    #     metrics=ragas_metrics,
    #     run_config=RunConfig(timeout=600, max_workers=1)
    # )

    ragas_df = results.to_pandas()

    retrieval_metrics = calculate_retrieval_metrics(
        hybrid_retriever, EVAL_SAMPLES
    )

    retrieval_score = (
        retrieval_metrics["Precision@5"]
        + retrieval_metrics["Recall@5"]
        + retrieval_metrics["MRR"]
    ) / 3

    ragas_score = ragas_df[
        ["faithfulness", "answer_relevancy",
         "context_precision", "context_recall"]
    ].mean().mean()

    overall_score = 0.3 * retrieval_score + 0.7 * ragas_score

    weight_results.append({
        "BM25 Weight": bm25_weight,
        "Semantic Weight": semantic_weight,
        **retrieval_metrics,
        "Faithfulness": ragas_df["faithfulness"].mean(),
        "Answer Relevancy": ragas_df["answer_relevancy"].mean(),
        "Context Precision": ragas_df["context_precision"].mean(),
        "Context Recall": ragas_df["context_recall"].mean(),
        "Retrieval Score": retrieval_score,
        "RAGAS Score": ragas_score,
        "Overall Score": overall_score
    })

weight_comparison = pd.DataFrame(weight_results).sort_values(
    "Overall Score", ascending=False
).reset_index(drop=True)

display(weight_comparison.round(4))


Testing weights: BM25=0.3, Semantic=0.7


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]


Testing weights: BM25=0.4, Semantic=0.6


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]


Testing weights: BM25=0.5, Semantic=0.5


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

... (2 outputs left out)


Testing weights: BM25=0.7, Semantic=0.3


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[8]: OutputParserException(Failed to parse StringIO from completion {"statements": [{"statement": "The key advantage of Retrieval-Augmented Generation is that it generates more factual language.", "reason": "The context explicitly states in the Discussion section that 'RAG models generate more specific, diverse and factual language than a state-of-the-art parametric-only baseline.' This directly supports the inference that this capability constitutes the key advantage. Verdict: 1", "verdict": 1}, {"statement": "This capability allows for specific and diverse output compared to purely parametric models.", "reason": "The context states that RAG generates 'more specific, diverse and factual language' compared to a state-of-the-art parametric-only baseline. This directly implies that the capability of RAG results in more specific and diverse output relative to parametric models. Verdict: 1", "verdict": 1}, {"statement": "RAG enables easy knowledg

,BM25 Weight,Semantic Weight,Precision@5,Recall@5,MRR,Faithfulness,Answer Relevancy,Context Precision,Context Recall,Retrieval Score,RAGAS Score,Overall Score
0,0.6,0.4,1.00,1.0,1.0,0.6250,0.9596,0.4933,0.75,1.0000,0.7070,0.7949
1,0.5,0.5,0.95,1.0,1.0,0.5801,0.9562,0.5097,0.75,0.9833,0.6990,0.7843
2,0.4,0.6,0.90,1.0,1.0,0.6958,0.9506,0.4248,0.75,0.9667,0.7053,0.7837
3,0.3,0.7,0.90,1.0,1.0,0.6958,0.9443,0.4165,0.75,0.9667,0.7017,0.7812
4,0.7,0.3,1.00,1.0,1.0,0.5000,0.9686,0.4933,0.75,1.0000,0.6780,0.7746


## Avanced RAG VS Basic RAG

In [59]:
from datasets import Dataset

basic_eval_data = []
advanced_eval_data = []

bm25_retriever1 = BM25Retriever.from_documents(all_documents, k=3)
semantic_retriever1 = vectorstore.as_retriever(search_kwargs={"k": 3})
hybrid_retriever1 = EnsembleRetriever(
    retrievers=[bm25_retriever1, semantic_retriever1],
    weights=[0.6, 0.4],
)

advanced_rag_chain1 =(
    {"context" : hybrid_retriever1 | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

for sample in EVAL_SAMPLES:

    q = sample["question"]
    ground_truth = sample["ground_truth"]
    basic_docs = vectorstore.similarity_search(q, k=3)
    basic_contexts = [doc.page_content for doc in basic_docs]
    basic_answer = basic_rag_chain.invoke(q)
    basic_eval_data.append({
        "user_input": q,
        "retrieved_contexts": basic_contexts,
        "response": basic_answer,
        "reference": ground_truth
    })

    advanced_docs = hybrid_retriever1.invoke(q)
    advanced_contexts = [doc.page_content for doc in advanced_docs]
    advanced_answer = advanced_rag_chain1.invoke(q)
    advanced_eval_data.append({
        "user_input": q,
        "retrieved_contexts": advanced_contexts,
        "response": advanced_answer,
        "reference": ground_truth
    })

print("Basic RAG samples:", len(basic_eval_data))
print("Advanced RAG samples:", len(advanced_eval_data))

Basic RAG samples: 4
Advanced RAG samples: 4


In [60]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

for metric in metrics:
    metric.llm = ragas_llm
    metric.embeddings = ragas_emb

basic_result = evaluate(
    Dataset.from_list(basic_eval_data),
    metrics=metrics
)

advanced_result = evaluate(
    Dataset.from_list(advanced_eval_data),
    metrics=metrics
)

/tmp/ipykernel_308/119478635.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_308/119478635.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/tmp/ipykernel_308/119478635.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
/tmp/ipykernel_308/119478635.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. 

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[12]: OutputParserException(Failed to parse StringIO from completion {"statements": [{"statement": "The original Transformer model was trained using an optimizer.", "reason": "The context explicitly states, 'We used the Adam optimizer' and discusses training of models, implying the use of an optimizer. This is a direct inference.", "verdict": 1}, {"statement": "The specific optimizer used is Adam.", "reason": "The text clearly states, 'We used the Adam optimizer', making this a direct inference.", "verdict": 1}, {"statement": "Adam optimizer parameters include beta_1 set to zero point nine.", "reason": "The context specifies 'β1 = 0.9' under the description of the Adam optimizer, which is a direct match.", "verdict": 1}, {"statement": "Adam optimizer parameters include beta_2 set to zero point nine eight.", "reason": "The context specifies 'β2 = 0.98' under the description of the Adam optimizer, which is a direct match.", "verdict": 1}, {"sta

In [61]:
import pandas as pd

metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

basic_scores = basic_result.to_pandas()[metric_names].mean()
advanced_scores = advanced_result.to_pandas()[metric_names].mean()

comparison = pd.DataFrame({
    "Basic RAG": basic_scores,
    "Advanced RAG": advanced_scores
})

comparison["Absolute Improvement"] = (
    comparison["Advanced RAG"] - comparison["Basic RAG"]
)

comparison["Relative Improvement (%)"] = (
    comparison["Absolute Improvement"]
    / comparison["Basic RAG"].replace(0, float("nan"))
) * 100

print(comparison.round(4))

                   Basic RAG  Advanced RAG  Absolute Improvement  \
faithfulness          0.4125        0.7500                0.3375   
answer_relevancy      0.7104        0.9416                0.2312   
context_precision     0.4167        0.5778                0.1611   
context_recall        0.5000        0.7500                0.2500   

                   Relative Improvement (%)  
faithfulness                        81.8182  
answer_relevancy                    32.5453  
context_precision                   38.6667  
context_recall                      50.0000  
